In [1]:
!pip install -q transformers datasets evaluate accelerate sentencepiece rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import shutil
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
import evaluate

In [3]:
MODEL_NAME = "facebook/bart-base"

INPUT_COL = "Report 1_Narrative"
TARGET_COL = "Report 1.2_Synopsis"

OUTPUT_DIR = "/kaggle/working/bart_task1_improved"

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 72

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)

print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)

Device: cuda
Output dir: /kaggle/working/bart_task1_improved


In [4]:
dataset = load_dataset("elihoole/asrs-aviation-reports")

print(dataset)
print(dataset["train"].column_names)

README.md: 0.00B [00:00, ?B/s]

asrs-aviation-reports-train.jsonl:   0%|          | 0.00/270M [00:00<?, ?B/s]

asrs-aviation-reports-validation.jsonl:   0%|          | 0.00/29.9M [00:00<?, ?B/s]

asrs-aviation-reports-test.jsonl:   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38655 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4295 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4773 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['acn_num_ACN', 'Time_Date', 'Time.1_Local Time Of Day', 'Place_Locale Reference', 'Place.1_State Reference', 'Place.2_Relative Position.Angle.Radial', 'Place.3_Relative Position.Distance.Nautical Miles', 'Place.4_Altitude.AGL.Single Value', 'Place.5_Altitude.MSL.Single Value', 'Environment_Flight Conditions', 'Environment.1_Weather Elements / Visibility', 'Environment.2_Work Environment Factor', 'Environment.3_Light', 'Environment.4_Ceiling', 'Environment.5_RVR.Single Value', 'Aircraft 1_ATC / Advisory', 'Aircraft 1.1_Aircraft Operator', 'Aircraft 1.2_Make Model Name', 'Aircraft 1.3_Aircraft Zone', 'Aircraft 1.4_Crew Size', 'Aircraft 1.5_Operating Under FAR Part', 'Aircraft 1.6_Flight Plan', 'Aircraft 1.7_Mission', 'Aircraft 1.8_Nav In Use', 'Aircraft 1.9_Flight Phase', 'Aircraft 1.10_Route In Use', 'Aircraft 1.11_Airspace', 'Aircraft 1.12_Maintenance Status.Maintenance Deferred', 'Aircraft 1.13_Maintenance Status.Records Complete',

In [5]:
def clean_split(ds):
    ds = ds.filter(
        lambda x: x[INPUT_COL] is not None and x[TARGET_COL] is not None
    )
    ds = ds.filter(
        lambda x: len(str(x[INPUT_COL]).strip()) > 20 and len(str(x[TARGET_COL]).strip()) > 5
    )
    return ds

dataset["train"] = clean_split(dataset["train"])
dataset["validation"] = clean_split(dataset["validation"])
dataset["test"] = clean_split(dataset["test"])

print(dataset)

Filter:   0%|          | 0/38655 [00:00<?, ? examples/s]

Filter:   0%|          | 0/38655 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4295 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4295 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4773 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4773 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['acn_num_ACN', 'Time_Date', 'Time.1_Local Time Of Day', 'Place_Locale Reference', 'Place.1_State Reference', 'Place.2_Relative Position.Angle.Radial', 'Place.3_Relative Position.Distance.Nautical Miles', 'Place.4_Altitude.AGL.Single Value', 'Place.5_Altitude.MSL.Single Value', 'Environment_Flight Conditions', 'Environment.1_Weather Elements / Visibility', 'Environment.2_Work Environment Factor', 'Environment.3_Light', 'Environment.4_Ceiling', 'Environment.5_RVR.Single Value', 'Aircraft 1_ATC / Advisory', 'Aircraft 1.1_Aircraft Operator', 'Aircraft 1.2_Make Model Name', 'Aircraft 1.3_Aircraft Zone', 'Aircraft 1.4_Crew Size', 'Aircraft 1.5_Operating Under FAR Part', 'Aircraft 1.6_Flight Plan', 'Aircraft 1.7_Mission', 'Aircraft 1.8_Nav In Use', 'Aircraft 1.9_Flight Phase', 'Aircraft 1.10_Route In Use', 'Aircraft 1.11_Airspace', 'Aircraft 1.12_Maintenance Status.Maintenance Deferred', 'Aircraft 1.13_Maintenance Status.Records Complete',

In [6]:
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [7]:
AIRCRAFT_PATTERNS = [
    r"\bB737[- ]?800\b", r"\bB737[- ]?700\b", r"\bB737[- ]?900\b",
    r"\b737[- ]?800\b", r"\b737[- ]?700\b", r"\b737[- ]?900\b",
    r"\bBoeing 737[- ]?800\b", r"\bBoeing 737[- ]?700\b", r"\bBoeing 737[- ]?900\b",
    r"\bCRJ[- ]?200\b", r"\bCRJ[- ]?700\b", r"\bCRJ[- ]?900\b",
    r"\bMD[- ]?11\b", r"\bMD11\b",
    r"\bSD3\b",
    r"\bC[- ]?162\b", r"\bC[- ]?172\b", r"\bCessna 162\b", r"\bCessna 172\b",
    r"\bA320\b", r"\bA321\b", r"\bA330\b", r"\bA350\b",
    r"\bB757\b", r"\bB767\b", r"\bB777\b", r"\bB787\b"
]

ROLE_PATTERNS = [
    r"\bCheck Pilot\b",
    r"\bFirst Officer\b",
    r"\bCaptain\b",
    r"\bTower Controller\b",
    r"\bTRACON Controller\b",
    r"\bDeparture Controller\b",
    r"\bController\b",
    r"\bFlight Crew\b",
    r"\bPilot\b"
]

ALTITUDE_PATTERNS = [
    r"\bFL ?\d{2,3}\b",
    r"\bflight level ?\d{2,3}\b",
    r"\b\d{3,5}\s?ft\b",
    r"\b\d{3,5}\s?feet\b",
    r"\b\d{3,5}\s?MSL\b"
]

RUNWAY_PATTERNS = [
    r"\bRWY\s?\d{1,2}[LRC]?\b",
    r"\bRunway\s\d{1,2}[LRC]?\b"
]

AIRPORT_PATTERNS = [
    r"\bK[A-Z]{3}\b",
    r"\b[A-Z]{3}\b airport\b"
]

PHASE_KEYWORDS = [
    "taxi", "takeoff", "departure", "climb",
    "cruise", "descent", "approach", "landing"
]

OBSERVATION_PATTERNS = [
    (r"\bsmoke\b", "smoke"),
    (r"\bturbulence\b", "turbulence"),
    (r"\bTCAS\b|\bRA\b", "TCAS alert"),
    (r"\bloss of separation\b", "loss of separation"),
    (r"\brunway incursion\b", "runway incursion"),
    (r"\bgo[- ]around\b", "go-around"),
    (r"\baltitude deviation\b|\bdeviat(ed|ion).{0,20}altitude\b", "altitude deviation"),
    (r"\bengine\b", "engine issue"),
    (r"\bequipment\b|\bmalfunction\b|\bfailure\b", "equipment malfunction"),
    (r"\bweather\b|\bwind shear\b|\bicing\b", "weather issue"),
    (r"\btraffic\b|\bconflict\b", "traffic conflict")
]

def extract_first_match(text, patterns):
    if text is None:
        return None
    text = str(text)
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(0)
    return None

def normalize_role(text):
    if text is None:
        return None
    t = text.lower().strip()
    mapping = {
        "check pilot": "Check Pilot",
        "first officer": "First Officer",
        "captain": "Captain",
        "tower controller": "Tower Controller",
        "tracon controller": "TRACON Controller",
        "departure controller": "Departure Controller",
        "controller": "Controller",
        "flight crew": "Flight Crew",
        "pilot": "Pilot",
    }
    return mapping.get(t, text)

def normalize_aircraft(text):
    if text is None:
        return None
    t = text.upper().replace(" ", "").replace("-", "")
    mapping = {
        "BOEING737800": "B737-800",
        "BOEING737700": "B737-700",
        "BOEING737900": "B737-900",
        "737800": "B737-800",
        "737700": "B737-700",
        "737900": "B737-900",
        "CRJ200": "CRJ-200",
        "CRJ700": "CRJ-700",
        "CRJ900": "CRJ-900",
        "MD11": "MD-11",
        "C162": "C-162",
        "C172": "C-172",
        "CESSNA162": "C-162",
        "CESSNA172": "C-172",
        "A320": "A320",
        "A321": "A321",
        "A330": "A330",
        "A350": "A350",
        "B757": "B757",
        "B767": "B767",
        "B777": "B777",
        "B787": "B787",
        "SD3": "SD3",
    }
    return mapping.get(t, text)

def normalize_altitude(text):
    if text is None:
        return None
    return str(text).upper().replace("FL ", "FL").strip()

def normalize_runway(text):
    if text is None:
        return None
    return str(text).upper().replace("RUNWAY", "RWY").replace("  ", " ").strip()

def normalize_airport(text):
    if text is None:
        return None
    t = str(text).strip()
    m = re.search(r"\b([A-Z]{3})\b", t)
    if m:
        return m.group(1)
    return t

def extract_aircraft_type(text):
    return normalize_aircraft(extract_first_match(text, AIRCRAFT_PATTERNS))

def extract_role(text):
    return normalize_role(extract_first_match(text, ROLE_PATTERNS))

def extract_altitude(text):
    return normalize_altitude(extract_first_match(text, ALTITUDE_PATTERNS))

def extract_runway(text):
    return normalize_runway(extract_first_match(text, RUNWAY_PATTERNS))

def extract_airport(text):
    return normalize_airport(extract_first_match(text, AIRPORT_PATTERNS))

def extract_phase(text):
    if text is None:
        return None
    t = str(text).lower()
    for kw in PHASE_KEYWORDS:
        if kw in t:
            return kw
    return None

def extract_observation(text):
    if text is None:
        return None
    t = str(text)
    for pattern, label in OBSERVATION_PATTERNS:
        if re.search(pattern, t, flags=re.IGNORECASE):
            return label
    return None

In [8]:
def split_sentences(text):
    text = str(text).strip()
    sents = re.split(r'(?<=[\.\!\?])\s+', text)
    return [s.strip() for s in sents if s.strip()]

def contains_critical_info(sent):
    sent_low = sent.lower()
    critical_terms = [
        "rwy", "runway", "airport", "approach", "landing", "descent", "climb",
        "captain", "first officer", "controller", "tower", "tracon", "departure",
        "fl", " ft", "feet", "tcas", "ra", "smoke", "traffic", "conflict",
        "go-around", "engine", "failure", "malfunction", "turbulence"
    ]
    if any(term in sent_low for term in critical_terms):
        return True
    if extract_aircraft_type(sent) or extract_altitude(sent) or extract_runway(sent):
        return True
    return False

def compress_narrative(text, head_chars=1800, tail_chars=700, max_critical_sents=4):
    text = str(text).strip()
    sents = split_sentences(text)

    critical_sents = []
    for s in sents:
        if contains_critical_info(s):
            critical_sents.append(s)
        if len(critical_sents) >= max_critical_sents:
            break

    head = text[:head_chars]
    tail = text[-tail_chars:] if len(text) > tail_chars else text

    parts = [head]
    if critical_sents:
        parts.append(" ".join(critical_sents))
    if len(text) > head_chars + tail_chars:
        parts.append(tail)

    combined = " ... ".join(parts)
    return re.sub(r"\s+", " ", combined).strip()

In [9]:
def safe_value(x):
    return x if x is not None else "unknown"

def build_prefix(narrative):
    aircraft = safe_value(extract_aircraft_type(narrative))
    role = safe_value(extract_role(narrative))
    observation = safe_value(extract_observation(narrative))
    phase = safe_value(extract_phase(narrative))
    altitude = safe_value(extract_altitude(narrative))
    runway = safe_value(extract_runway(narrative))
    airport = safe_value(extract_airport(narrative))

    return (
        f"aircraft: {aircraft}; "
        f"role: {role}; "
        f"observation: {observation}; "
        f"phase: {phase}; "
        f"altitude: {altitude}; "
        f"runway: {runway}; "
        f"airport: {airport}; "
        f"summarize: "
    )

In [10]:
def preprocess_function(examples):
    raw_inputs = [str(x).strip() for x in examples[INPUT_COL]]
    targets = [str(y).strip() for y in examples[TARGET_COL]]

    inputs = [
        build_prefix(x) + compress_narrative(x)
        for x in raw_inputs
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]
    labels_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

In [11]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset

Map:   0%|          | 0/38651 [00:00<?, ? examples/s]

Map:   0%|          | 0/4295 [00:00<?, ? examples/s]

Map:   0%|          | 0/4773 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 38651
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4295
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4773
    })
})

In [12]:
rouge = evaluate.load("rouge")

In [13]:
def key_info_scores(decoded_preds, decoded_labels):
    aircraft_scores = []
    role_scores = []
    altitude_scores = []
    runway_scores = []
    airport_scores = []
    observation_scores = []

    for pred, ref in zip(decoded_preds, decoded_labels):
        pred_aircraft = extract_aircraft_type(pred)
        ref_aircraft = extract_aircraft_type(ref)

        pred_role = extract_role(pred)
        ref_role = extract_role(ref)

        pred_altitude = extract_altitude(pred)
        ref_altitude = extract_altitude(ref)

        pred_runway = extract_runway(pred)
        ref_runway = extract_runway(ref)

        pred_airport = extract_airport(pred)
        ref_airport = extract_airport(ref)

        pred_observation = extract_observation(pred)
        ref_observation = extract_observation(ref)

        aircraft_scores.append(float(pred_aircraft == ref_aircraft) if ref_aircraft else np.nan)
        role_scores.append(float(pred_role == ref_role) if ref_role else np.nan)
        altitude_scores.append(float(pred_altitude == ref_altitude) if ref_altitude else np.nan)
        runway_scores.append(float(pred_runway == ref_runway) if ref_runway else np.nan)
        airport_scores.append(float(pred_airport == ref_airport) if ref_airport else np.nan)
        observation_scores.append(float(pred_observation == ref_observation) if ref_observation else np.nan)

    def nanmean_safe(x):
        x = np.array(x, dtype=np.float32)
        return float(np.nanmean(x)) if np.any(~np.isnan(x)) else 0.0

    return {
        "aircraft_precision": nanmean_safe(aircraft_scores),
        "role_precision": nanmean_safe(role_scores),
        "altitude_precision": nanmean_safe(altitude_scores),
        "runway_precision": nanmean_safe(runway_scores),
        "airport_precision": nanmean_safe(airport_scores),
        "observation_precision": nanmean_safe(observation_scores),
    }

def concision_score(decoded_preds, min_words=12, max_words=35):
    scores = []
    lengths = []

    for pred in decoded_preds:
        n = len(pred.split())
        lengths.append(n)

        if min_words <= n <= max_words:
            scores.append(1.0)
        elif n < min_words:
            scores.append(max(0.0, n / min_words))
        else:
            overflow = n - max_words
            scores.append(max(0.0, 1.0 - overflow / max_words))

    return float(np.mean(scores)), float(np.mean(lengths))

In [14]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    predictions = predictions.astype(np.int64)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id).astype(np.int64)
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    ki = key_info_scores(decoded_preds, decoded_labels)
    concision, avg_len = concision_score(decoded_preds)

    faithfulness = np.mean([
        ki["aircraft_precision"],
        ki["role_precision"],
        ki["altitude_precision"],
        ki["runway_precision"],
        ki["airport_precision"],
        ki["observation_precision"],
    ])

    final_score = (
        0.60 * faithfulness +
        0.20 * rouge_scores["rougeL"] +
        0.20 * concision
    )

    return {
        "rouge1": round(rouge_scores["rouge1"] * 100, 2),
        "rouge2": round(rouge_scores["rouge2"] * 100, 2),
        "rougeL": round(rouge_scores["rougeL"] * 100, 2),

        "aircraft_precision": round(ki["aircraft_precision"] * 100, 2),
        "role_precision": round(ki["role_precision"] * 100, 2),
        "altitude_precision": round(ki["altitude_precision"] * 100, 2),
        "runway_precision": round(ki["runway_precision"] * 100, 2),
        "airport_precision": round(ki["airport_precision"] * 100, 2),
        "observation_precision": round(ki["observation_precision"] * 100, 2),

        "faithfulness": round(faithfulness * 100, 2),
        "concision": round(concision * 100, 2),
        "avg_pred_len": round(avg_len, 2),

        "final_score": round(final_score * 100, 2),
    }

In [15]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,

    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    weight_decay=0.01,
    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="final_score",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [16]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Aircraft Precision,Role Precision,Altitude Precision,Runway Precision,Airport Precision,Observation Precision,Faithfulness,Concision,Avg Pred Len,Final Score
1,10.278708,2.363632,31.510000,12.910000,27.040000,14.410000,49.470000,9.240000,18.810000,12.800000,39.150000,23.980000,95.070000,14.500000,38.810000
2,9.483243,2.260386,32.920000,13.780000,28.070000,17.540000,50.160000,14.520000,14.220000,8.530000,41.720000,24.450000,96.080000,15.220000,39.500000
3,9.145756,2.229191,33.560000,14.150000,28.610000,20.220000,50.530000,14.190000,14.220000,9.480000,44.150000,25.470000,95.840000,15.640000,40.170000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=7248, training_loss=10.027354166734034, metrics={'train_runtime': 8313.9927, 'train_samples_per_second': 13.947, 'train_steps_per_second': 0.872, 'total_flos': 3.535038577115136e+16, 'train_loss': 10.027354166734034, 'epoch': 3.0})

In [18]:
val_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"])

print("Validation metrics:")
print(val_metrics)

print("\nTest metrics:")
print(test_metrics)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation metrics:
{'eval_loss': 2.229191303253174, 'eval_rouge1': 33.56, 'eval_rouge2': 14.15, 'eval_rougeL': 28.61, 'eval_aircraft_precision': 20.22, 'eval_role_precision': 50.53, 'eval_altitude_precision': 14.19, 'eval_runway_precision': 14.22, 'eval_airport_precision': 9.48, 'eval_observation_precision': 44.15, 'eval_faithfulness': 25.47, 'eval_concision': 95.84, 'eval_avg_pred_len': 15.64, 'eval_final_score': 40.17, 'eval_runtime': 690.9989, 'eval_samples_per_second': 6.216, 'eval_steps_per_second': 0.777, 'epoch': 3.0}

Test metrics:
{'eval_loss': 2.2063512802124023, 'eval_rouge1': 34.0, 'eval_rouge2': 14.66, 'eval_rougeL': 29.14, 'eval_aircraft_precision': 21.73, 'eval_role_precision': 49.81, 'eval_altitude_precision': 15.32, 'eval_runway_precision': 13.57, 'eval_airport_precision': 8.44, 'eval_observation_precision': 44.73, 'eval_faithfulness': 25.6, 'eval_concision': 96.02, 'eval_avg_pred_len': 15.58, 'eval_final_score': 40.39, 'eval_runtime': 762.9937, 'eval_samples_per_seco

In [19]:
GEN_KWARGS = dict(
    max_new_tokens=64,
    min_new_tokens=12,
    num_beams=6,
    length_penalty=0.95,
    no_repeat_ngram_size=3,
    early_stopping=True
)

def generate_summary(text):
    text_in = build_prefix(text) + compress_narrative(text)

    inputs = tokenizer(
        text_in,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(DEVICE)

    generated_ids = model.generate(**inputs, **GEN_KWARGS)
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [20]:
for i in range(5):
    narrative = dataset["test"][i][INPUT_COL]
    reference = dataset["test"][i][TARGET_COL]
    prediction = generate_summary(narrative)

    print("=" * 100)
    print("NARRATIVE:\n", narrative[:1200], "...\n")
    print("REFERENCE:\n", reference, "\n")
    print("PREDICTION:\n", prediction, "\n")

NARRATIVE:
 We were on approach to landing Runway 27. I was pilot flying. The approach was normal; in the flare I brought the thrust levers to idle at the appropriate time. The left thrust lever stopped at idle and the right continued slightly. When I went to deploy reverse thrust; the left reverser deployed and the right thrust lever went back a little bit more. The Captain took control of the airplane when only the left reverser deployed. After we were pulled off the runway; I saw the right engine was in shutdown and the engine was spooling down. We stopped on the parallel taxiway and restarted the engine before taxiing back to the gate. ...

REFERENCE:
 A First Officer and Check Pilot reported a sudden; unexplained shutdown of the right engine on a CRJ-700 aircraft during the landing rollout on a training flight. The Check Pilot noted he never saw the FO pull the right engine shut-off trigger and only observed the First Officer's hand on top of the power levers. 

PREDICTION:
 B737-

In [21]:
rows = []

N_TEST = min(300, len(dataset["test"]))

for i in range(N_TEST):
    narrative = dataset["test"][i][INPUT_COL]
    reference = dataset["test"][i][TARGET_COL]
    prediction = generate_summary(narrative)

    rows.append({
        "narrative": narrative,
        "reference_synopsis": reference,
        "predicted_synopsis": prediction,

        "ref_aircraft": extract_aircraft_type(reference),
        "pred_aircraft": extract_aircraft_type(prediction),

        "ref_role": extract_role(reference),
        "pred_role": extract_role(prediction),

        "ref_altitude": extract_altitude(reference),
        "pred_altitude": extract_altitude(prediction),

        "ref_runway": extract_runway(reference),
        "pred_runway": extract_runway(prediction),

        "ref_airport": extract_airport(reference),
        "pred_airport": extract_airport(prediction),

        "ref_observation": extract_observation(reference),
        "pred_observation": extract_observation(prediction),

        "pred_len_words": len(prediction.split()),
    })

df_eval = pd.DataFrame(rows)

for col in ["aircraft", "role", "altitude", "runway", "airport", "observation"]:
    df_eval[f"{col}_match"] = df_eval[f"ref_{col}"] == df_eval[f"pred_{col}"]

df_eval["faithfulness_row"] = df_eval[
    ["aircraft_match", "role_match", "altitude_match", "runway_match", "airport_match", "observation_match"]
].mean(axis=1)

df_eval["concision_ok"] = df_eval["pred_len_words"].between(12, 35)

print("Aircraft precision:", df_eval["aircraft_match"].mean())
print("Role precision:", df_eval["role_match"].mean())
print("Altitude precision:", df_eval["altitude_match"].mean())
print("Runway precision:", df_eval["runway_match"].mean())
print("Airport precision:", df_eval["airport_match"].mean())
print("Observation precision:", df_eval["observation_match"].mean())
print("Faithfulness mean:", df_eval["faithfulness_row"].mean())
print("Concision ok rate:", df_eval["concision_ok"].mean())
print("Average predicted length:", df_eval["pred_len_words"].mean())

Aircraft precision: 0.05333333333333334
Role precision: 0.37333333333333335
Altitude precision: 0.016666666666666666
Runway precision: 0.0
Airport precision: 0.0033333333333333335
Observation precision: 0.13666666666666666
Faithfulness mean: 0.09722222222222221
Concision ok rate: 0.7566666666666667
Average predicted length: 15.546666666666667


In [22]:
df_eval_sorted = df_eval.sort_values(by="faithfulness_row", ascending=True)

df_eval_sorted[[
    "reference_synopsis",
    "predicted_synopsis",
    "ref_aircraft", "pred_aircraft",
    "ref_role", "pred_role",
    "ref_altitude", "pred_altitude",
    "ref_runway", "pred_runway",
    "ref_airport", "pred_airport",
    "ref_observation", "pred_observation",
    "faithfulness_row",
    "pred_len_words"
]].head(20)

,reference_synopsis,predicted_synopsis,ref_aircraft,pred_aircraft,ref_role,pred_role,ref_altitude,pred_altitude,ref_runway,pred_runway,ref_airport,pred_airport,ref_observation,pred_observation,faithfulness_row,pred_len_words
299,A Safety Manager reported the crew of a Gulfst...,G650ER flight crew reported diverting to an al...,None,None,None,Flight Crew,None,None,None,None,None,None,None,None,0.0,13
281,Pilot requests deviation heading for weather a...,B737 Captain reported being unable to maintain...,None,None,Controller,Captain,None,None,None,None,None,None,weather issue,None,0.0,19
282,Flight of two F/A-18s took evasive action to a...,C172 instructor pilot reported a near midair c...,None,C-172,None,Pilot,None,None,None,None,None,None,None,None,0.0,17
297,Four reports from an SCT Controller providing ...,SCT TRACON Controller reported an NMAC with an...,None,None,Controller,TRACON Controller,None,None,None,None,None,None,None,None,0.0,10
296,The reporter expressed concern that during win...,Air carrier Captain reported that there is a s...,None,None,None,Captain,None,None,None,None,None,None,None,None,0.0,20
295,Pilot reported going to the de-ice pad; but no...,Air carrier Captain reported a taxiway incursi...,None,None,Pilot,Captain,None,None,None,None,None,CDF,None,None,0.0,15
30,A321 First Officer reported both sets of stabi...,B737-800 flight crew reported stabilizer trim ...,A321,B737-800,First Officer,Flight Crew,None,None,None,None,None,None,None,None,0.0,9
16,B757 Captain reported a track deviation occurr...,B737-700 flight crew reported a track deviatio...,B757,B737-700,Captain,Flight Crew,None,None,None,RWY 26R,None,None,None,None,0.0,17
20,Air Carrier Captain reported the van driver fr...,Flight Attendant reported a passenger did not ...,None,None,Captain,None,None,None,None,None,the airport,None,None,None,0.0,12
23,A Dispatcher related concern regarding routine...,Air carrier Captain reported loss of communica...,None,None,None,Captain,None,None,None,None,None,None,None,None,0.0,15


In [23]:
manual_rows = []

for i in range(min(30, len(dataset["test"]))):
    narrative = dataset["test"][i][INPUT_COL]
    reference = dataset["test"][i][TARGET_COL]
    prediction = generate_summary(narrative)

    manual_rows.append({
        "id": i,
        "reference": reference,
        "prediction": prediction,
        "faithful": "",
        "concise": "",
        "critical_error": "",
        "missing_key_info": "",
        "notes": ""
    })

manual_df = pd.DataFrame(manual_rows)
manual_path = f"{OUTPUT_DIR}/manual_review.csv"
manual_df.to_csv(manual_path, index=False)

print("Saved:", manual_path)
manual_df.head()

Saved: /kaggle/working/bart_task1_improved/manual_review.csv


,id,reference,prediction,faithful,concise,critical_error,missing_key_info,notes
0,0,A First Officer and Check Pilot reported a sud...,B737-800 flight crew reported the right engine...,,,,,
1,1,SD3 First Officer reports loosing 400 feet fro...,Air carrier First Officer reported a loss of a...,,,,,
2,2,MD11 flight crew experiences an Air System 3 l...,B737-800 flight crew reported receiving a Leve...,,,,,
3,3,C162 pilot reported contacting the wingtip of ...,Pilot reported a ground conflict with a jet ai...,,,,,
4,4,B737-700 Captain reported a track deviation oc...,B737 flight crew reported encountering wake tu...,,,,,


In [24]:
with open(f"{OUTPUT_DIR}/validation_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=2)

with open(f"{OUTPUT_DIR}/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

df_eval.to_csv(f"{OUTPUT_DIR}/detailed_test_eval.csv", index=False)

trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_tokenizer")

print("Saved all outputs to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved all outputs to: /kaggle/working/bart_task1_improved


In [25]:
zip_base = "/kaggle/working/bart_task1_improved_results"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Created:", zip_base + ".zip")

Created: /kaggle/working/bart_task1_improved_results.zip


In [26]:
def replace_aircraft_if_needed(narrative, prediction):
    true_aircraft = extract_aircraft_type(narrative)
    pred_aircraft = extract_aircraft_type(prediction)

    if true_aircraft and pred_aircraft and true_aircraft != pred_aircraft:
        prediction = re.sub(
            re.escape(pred_aircraft),
            true_aircraft,
            prediction,
            count=1,
            flags=re.IGNORECASE
        )
    return prediction

def replace_role_if_needed(narrative, prediction):
    true_role = extract_role(narrative)
    pred_role = extract_role(prediction)

    if true_role and pred_role and true_role.lower() != pred_role.lower():
        prediction = re.sub(
            re.escape(pred_role),
            true_role,
            prediction,
            count=1,
            flags=re.IGNORECASE
        )
    return prediction

def correct_prediction(narrative, prediction):
    prediction = replace_aircraft_if_needed(narrative, prediction)
    prediction = replace_role_if_needed(narrative, prediction)
    return prediction

In [27]:
for i in range(5):
    narrative = dataset["test"][i][INPUT_COL]
    pred = generate_summary(narrative)
    corrected = correct_prediction(narrative, pred)

    print("=" * 100)
    print("RAW:      ", pred)
    print("CORRECTED:", corrected)

RAW:       B737-800 flight crew reported the right engine was in shutdown and the engine spooling down.
CORRECTED: B737-800 Captain reported the right engine was in shutdown and the engine spooling down.
RAW:       Air carrier First Officer reported a loss of altitude due to turbulence in the vicinity of thunderstorms.
CORRECTED: Air carrier Captain reported a loss of altitude due to turbulence in the vicinity of thunderstorms.
RAW:       B737-800 flight crew reported receiving a Level 2 'Air System Pressure 3 Lo' alert during approach.
CORRECTED: B737-800 Captain reported receiving a Level 2 'Air System Pressure 3 Lo' alert during approach.
RAW:       Pilot reported a ground conflict with a jet aircraft while taxiing to the end of the runway.
CORRECTED: Pilot reported a ground conflict with a jet aircraft while taxiing to the end of the runway.
RAW:       B737 flight crew reported encountering wake turbulence climbing through FL380.
CORRECTED: B737 flight crew reported encountering wa